## Validation of model_predict_cavity_claw_RouteMeander_eigenmode wtih Ansys HFSS

testing pipeline for quarter wavelength resonators only

In [1]:
from squadds import SQuADDS_DB
import pandas as pd
from squadds import Analyzer
import matplotlib.pyplot as plt
from squadds import AnsysSimulator
import numpy as np

We don't actually care about using SQuADDS to get a device at the moment, we just use the code block below to get a "template" of the SQUaDDS style dataframe.

In [2]:
''' grab SQuADDS entry as a template for ML predicted designs ''' 
db = SQuADDS_DB()
db.select_system(["qubit","cavity_claw"])
db.select_qubit("TransmonCross")
db.select_cavity_claw("RouteMeander")
db.select_resonator_type("quarter")
df = db.create_system_df()
analyzer = Analyzer(db)

# we are not actually looking for these Hamiltonian parameters... 
target_params={"qubit_frequency_GHz": 4,"anharmonicity_MHz": -200,"cavity_frequency_GHz":6.2,"kappa_kHz":20,"g_MHz":70}

pred_df = analyzer.find_closest(target_params=target_params,
                                       num_top=1,
                                       metric="Euclidean",
                                       display=True)

''' read in ML results ''' 
ML_results_total = pd.read_csv("predictions_and_errors_unscaled_one_hot.csv") # real in ML test results


''' for now we only want to consider predicted quarter wavelength resonators ''' 
ML_results = ML_results_total.iloc[0:0].copy()
for i in np.unique(df_unscaled.sample_idx):
    temp_df = ML_results_total[ML_results_total.sample_idx == i]

    if temp_df[temp_df.param == 'resonator_type_half'].ref_unscaled.iloc[0] == 1: # do not keep half wavelength resonators
        continue
    else:
        ML_results = pd.concat([ML_results, temp_df])

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SQuADDS/SQuADDS_DB/6b4001554a05a484bcbb5faa7363f9c3097d5174/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/6b4001554a05a484bcbb5faa7363f9c3097d5174/SQuADDS_DB.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SQuADDS/SQuADDS_DB/SQuADDS/SQuADDS_DB.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/SQuADDS/SQuADDS_DB/revision/6b4001554a05a484bcbb5faa7363f9c3097d5174 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/6b4001554a05a484bcbb5faa7363f9c3097d5174/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://

Time taken to add the coupled H params: 1.1261565685272217 seconds


## Sweep through ML design results, simulate each design with Anysys HFSS (eigenmode), and save sim results
Below are the multi-valued parameters in the SQuADDS cavity_claw-RouteMeander-eigenmode dataset:

* design_options.claw_opts.connection_pads.readout.claw_length
* design_options.claw_opts.connection_pads.readout.ground_spacing
* design_options.claw_opts.pos_x
* design_options.cplr_opts.coupling_length
* design_options.cpw_opts.lead.end_straight
* design_options.cpw_opts.lead.start_jogged_extension.0
* design_options.cpw_opts.lead.start_straight
* design_options.cpw_opts.meander.asymmetry
* design_options.cpw_opts.total_length

There are four (really only two) binary paramters as well:
* coupler_type_CLT
* coupler_type_NCap
* resonator_type_half
* resonator_type_quarter

Only two really because if resonator_type_quarter is true then the coupler type is by definition coupler_type_CLT, and similarly for resonator_type_half with coupler_type_NCap.

The model only predicts these multi-valued design parameters listed above - however, we only consider quarter wavelength resonators for now. We take a SQuADDS database entry and substitute in the predicted parameters from the model.

In [5]:
# dictionary to save results in 
results = pd.DataFrame({"Sample":[],
                   "ref_H_params":[],
                   "pred_H_params":[]})

um = 10**6 ## ML model is trained in SI units (m), convert back to µm  
samples_to_test = np.unique(ML_results.sample_idx)

In [6]:
for sample in samples_to_test:
        
    ''' current testing sample '''
    this_device = ML_results[ML_results.sample_idx == sample]

    ''' create our predicted design option for Qiskit Metal '''
    for param in np.unique(this_device.param):

        if int(this_device[this_device.param == param].exists_pred_mask) == 1:

            param_keys = param.split(".")[1:]
            param_value = str(float((this_device[this_device.param == param].pred_unscaled))*um)+"um"

            if len(param_keys) == 2:
                pred_df.design_options_cavity_claw.iloc[0][param_keys[0]][param_keys[1]] = param_value
            elif len(param_keys) == 3:
                pred_df.design_options_cavity_claw.iloc[0][param_keys[0]][param_keys[1]][param_keys[2]] = param_value
            elif len(param_keys) == 4:
                pred_df.design_options_cavity_claw.iloc[0][param_keys[0]][param_keys[1]][param_keys[2]][param_keys[3]] = param_value

    ''' simulate predicted design '''
    pred_ansys_simulator = AnsysSimulator(analyzer, pred_df.iloc[0])
    pred_ansys_results = pred_ansys_simulator.simulate(pred_df.iloc[0])

    pred_Hamiltonian_params = {'cavity_frequency_GHz':pred_ansys_results["sim_results"]["cavity_frequency_GHz"],
                               'kappa_kHz':pred_ansys_results["sim_results"]['kappa_kHz']}
    ref_Hamiltonian_params = {'cavity_frequency_GHz':this_device.cavity_frequency.iloc[0],
                               'kappa_kHz':this_device.kappa.iloc[0]}
    
    ''' save results '''
    results.loc[len(results)] = [sample,
                                 ref_Hamiltonian_params,
                                 pred_Hamiltonian_params]

 C:\Users\firas\AppData\Local\Temp\6\ipykernel_16776\943299996.py: 14
 C:\Users\firas\AppData\Local\Temp\6\ipykernel_16776\943299996.py: 17
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\qiskit_metal\qgeometries\qgeometries_handler.py: 528


selected system: ['qubit', 'cavity_claw']
{'aedt_hfss_capacitance': 0, 'aedt_hfss_inductance': 9.686e-09, 'aedt_q3d_capacitance': 0, 'aedt_q3d_inductance': 1e-08, 'chip': 'main', 'connection_pads': {'readout': {'claw_cpw_length': '40um', 'claw_cpw_width': '10um', 'claw_gap': '5.1um', 'claw_length': '130um', 'claw_width': '15um', 'connector_location': '90', 'connector_type': '0', 'ground_spacing': '4.1um'}}, 'cross_gap': '30um', 'cross_length': '200um', 'cross_width': '30um', 'gds_cell_name': 'my_other_junction', 'hfss_capacitance': 0, 'hfss_inductance': 9.686e-09, 'hfss_mesh_kw_jj': 7e-06, 'hfss_resistance': 0, 'layer': '1', 'orientation': '-90', 'pos_x': '-1500um', 'pos_y': '1200um', 'q3d_capacitance': 0, 'q3d_inductance': '10nH', 'q3d_mesh_kw_jj': 7e-06, 'q3d_resistance': 0}
Starting the Simulation


INFO 12:08AM [connect_project]: Connecting to Ansys Desktop API...
INFO 12:08AM [load_ansys_project]: 	Opened Ansys App
INFO 12:08AM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 12:08AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/firas/Documents/Ansoft/
	Project:   Project17
INFO 12:08AM [connect_design]: No active design found (or error getting active design).
INFO 12:08AM [connect]: 	 Connected to project "Project17". No design detected
INFO 12:08AM [connect_design]: 	Opened active design
	Design:    CavitySweep [Solution type: Eigenmode]
WARNING 12:08AM [connect_setup]: 	No design setup detected.
WARNING 12:08AM [connect_setup]: 	Creating eigenmode default setup.
INFO 12:08AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


the parameters ['min_converged_passes'] are unsupported, so they have been ignored


INFO 12:08AM [connect_design]: 	Opened active design
	Design:    CavitySweep_hfss [Solution type: Eigenmode]
WARNING 12:08AM [connect_setup]: 	No design setup detected.
WARNING 12:08AM [connect_setup]: 	Creating eigenmode default setup.
INFO 12:08AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


...

INFO:Global:Materials class has been initialized! Elapsed time: 0m 0sec


PyAEDT INFO: Desktop has been released.


INFO:Global:Desktop has been released.
INFO 10:16AM [__del__]: Disconnected from Ansys HFSS
INFO 10:16AM [__del__]: Disconnected from Ansys HFSS
INFO 10:16AM [get_setup]: 	Opened setup `test_setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 10:16AM [analyze]: Analyzing setup test_setup
10:59AM 42s INFO [get_f_convergence]: Saved convergences to C:\Users\firas\Desktop\ML for qubit design\cavity_claw\hfss_eig_f_convergence.csv


couldn't generate plots.


com_error: (-2147023174, 'The RPC server is unavailable.', None, None)

# Save the data

In [8]:
results.to_csv("cavityClaw_simulation_results.csv",index=False) # save data for later analysis with validation_01_data_analysis.ipynb